# Educational example 1 — carbon footprint of a cup of tea

The full workflow (presample → Monte Carlo → GSA) on a **self-contained** Brightway2 model: no ecoinvent, no project backup. Same machinery as the production `MC_workflow.ipynb`, but a model you can check by hand.

`GWI = (water_volume · elec_per_litre) · grid_CO₂ + teabag_CO₂`

> Run with a **numpy < 2** kernel.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import bw2data as bd, brightway2 as bw
from scipy.stats import norm, uniform, triang, lognorm, bernoulli, beta
from SALib.analyze import delta

assert int(np.__version__.split('.')[0]) < 2, 'use a numpy < 2 kernel (bw2data 3.6.x)'

## Layer 1 — build a minimal parameterised project

A one-flow biosphere (CO₂), a trivial GWP method (1 kg CO₂-eq per kg CO₂), and three activities. The `formula` key on an exchange means its amount is recomputed from the parameters each iteration.

In [ ]:
bd.projects.set_current('tea_demo')
bd.Database('demo_bio').write({('demo_bio','co2'):{'name':'carbon dioxide','categories':('air',),'type':'emission','unit':'kg'}})
m = bd.Method(('demo','GWP')); m.register(); m.write([(('demo_bio','co2'), 1.0)])

bd.Database('tea').write({
  ('tea','electricity'): {'name':'electricity, 1 kWh','unit':'kWh','exchanges':[
      {'input':('tea','electricity'),'amount':1,'type':'production'},
      {'input':('demo_bio','co2'),'amount':0.4,'type':'biosphere','formula':'grid_CO2'}]},
  ('tea','teabag'): {'name':'tea bag','unit':'unit','exchanges':[
      {'input':('tea','teabag'),'amount':1,'type':'production'},
      {'input':('demo_bio','co2'),'amount':0.002,'type':'biosphere','formula':'teabag_CO2'}]},
  ('tea','cup'): {'name':'cup of tea','unit':'cup','exchanges':[
      {'input':('tea','cup'),'amount':1,'type':'production'},
      {'input':('tea','electricity'),'amount':0.025,'type':'technosphere','formula':'water_volume * elec_per_litre'},
      {'input':('tea','teabag'),'amount':1,'type':'technosphere'}]},   # fixed, no formula
})
fu = bd.Database('tea').get('cup'); method = bd.Method(('demo','GWP'))

## Layer 2 — parameter registry

One row per uncertain parameter; `name` must match the formula variables.

In [ ]:
SAMPLERS = {
    'normal':     lambda size, loc, scale: norm.rvs(loc, scale, size=size),
    'uniform':    lambda size, low, width: uniform.rvs(low, width, size=size),   # support [low, low+width]
    'triangular': lambda size, c, loc, scale: triang.rvs(c, loc, scale, size=size),
    'lognormal':  lambda size, s, scale: lognorm.rvs(s, scale=scale, size=size),
}
PARAMETERS = [
    dict(name='water_volume',   dist='uniform',    args=dict(low=0.20, width=0.10),       desc='Water boiled (L)'),
    dict(name='elec_per_litre', dist='normal',     args=dict(loc=0.10, scale=0.012),      desc='Energy to boil 1 L (kWh)'),
    dict(name='grid_CO2',       dist='triangular', args=dict(c=0.43, loc=0.1, scale=0.7), desc='Grid intensity (kg CO2/kWh)'),
    dict(name='teabag_CO2',     dist='lognormal',  args=dict(s=np.log(1.3), scale=0.002), desc='Tea-bag CO2 (kg)'),
]

## Layer 3 — presampling

Draw N scenarios; fix the seed for reproducibility.

In [ ]:
np.random.seed(42); N = 5000
sampled = {}
for p in PARAMETERS:                       # draw in list order; a string arg = use an earlier sample
    a = {k:(sampled[v] if isinstance(v,str) else v) for k,v in p['args'].items()}
    sampled[p['name']] = SAMPLERS[p['dist']](size=N, **a)
names = [p['name'] for p in PARAMETERS]
X = np.column_stack([sampled[n] for n in names])
print('X shape:', X.shape)

## Layers 6–7 — the Monte Carlo engine

`run_mc` is identical to the production workflow: for each scenario it writes every `formula` exchange from that row's parameters and runs the LCA. Because **all** parameterised exchanges are overwritten every iteration, the result depends only on the sample matrix `X` — not on leftover database state.

In [ ]:
def run_mc(X, names, fexc, fu, method, n=None):
    n = n or len(X); out = np.empty(n)
    for i in range(n):
        p = dict(zip(names, X[i].tolist()))          # this scenario's parameter values
        for e in fexc:                                # overwrite every formula exchange
            e['amount'] = float(eval(e['formula'], {'__builtins__': None}, p)); e.save()
        lca = bw.LCA({fu: 1}, method.name); lca.lci(); lca.lcia(); out[i] = lca.score
    return out

fexc = [e for a in bd.Database('tea') for e in a.exchanges() if 'formula' in e]
Y = run_mc(X, names, fexc, fu, method)
print(f'mean = {Y.mean():.4g} kg CO2-eq / cup   P5 = {np.percentile(Y,5):.4g}   P95 = {np.percentile(Y,95):.4g}')

## Layer 8 — visualise the Monte Carlo output

Histogram + boxplot of the impact distribution (how uncertain the result is).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(Y, bins=40, color='#4c72b0'); ax[0].set_title('Distribution of the impact')
ax[0].set_xlabel('kg CO2-eq / cup'); ax[0].set_ylabel('count')
ax[1].boxplot(Y); ax[1].set_xticks([]); ax[1].set_title('Boxplot')
plt.tight_layout(); plt.show()

### Output vs. each input (scatter screening)

One scatter per parameter — the *shape* tells you how it acts: a clear trend = influential, a flat cloud = negligible, two vertical stripes = a binary on/off choice. This is the visual companion to the δ measure.

In [ ]:
ncol = 4; nrow = int(np.ceil(len(names)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3, nrow*2.6), sharey=True)
axes = np.atleast_1d(axes).flatten()
for j, nm in enumerate(names):
    axes[j].scatter(X[:, j], Y, s=5, alpha=0.2, color='#4c72b0')
    axes[j].set_xlabel(nm, fontsize=9)
for k in range(len(names), len(axes)): axes[k].axis('off')
fig.suptitle('Model output vs each input (kg CO2-eq / cup)'); plt.tight_layout(rect=[0,0,1,0.96]); plt.show()

## Layer 9 — global sensitivity analysis (Borgonovo δ)

δ ranks how strongly each input drives the output uncertainty (moment-independent, so it handles the binary choices too).

In [ ]:
problem = {'num_vars': len(names), 'names': names, 'bounds': list(zip(X.min(0), X.max(0)))}
df_gsa = delta.analyze(problem, X, Y).to_df().sort_values('delta')
print(df_gsa[['delta','S1']].round(4).iloc[::-1].to_string())

plt.figure(figsize=(7, 0.4*len(names)+1))
plt.barh(range(len(df_gsa)), df_gsa['delta'], color='#c44e52')
plt.yticks(range(len(df_gsa)), df_gsa.index); plt.xlabel('Borgonovo δ')
plt.title('Global sensitivity — which inputs drive the result'); plt.tight_layout(); plt.show()

## What to take away

- Mean ≈ **0.013 kg CO₂-eq/cup**.
- δ ranking: **`grid_CO2` ≫ `elec_per_litre` ≈ `water_volume` ≫ `teabag_CO2`**.
- The scatter for `grid_CO2` shows a strong trend; `teabag_CO2` is a flat cloud → negligible.
- **Lesson:** decarbonising the electricity matters far more than the tea bag. Try narrowing `grid_CO2`'s range and re-run — the ranking shifts, which is the whole point of GSA.